# Part 1 – Neural Network Fundamentals and Training Behavior Analysis

**Dataset:** [Customer Churn Neural Network Dataset](https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing)  
**Goal:** Predict whether a customer will churn (`churn = 1`) or be retained (`churn = 0`)  
**Problem Type:** Binary Classification

> The raw dataset is loaded as-is. No rows, columns, or values are modified.

---
## Table of Contents
- [Task 1: Dataset Understanding](#task-1)
- [Task 2: Data Preprocessing](#task-2)
- [Task 3: Neural Network Model Building](#task-3)
- [Task 4: Training and Evaluation](#task-4)
- [Task 5: Hyperparameter Experimentation](#task-5)
- [Task 6: Final Reflection](#task-6)


## Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, roc_curve
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "#F8FAFC", "axes.facecolor": "#F8FAFC",
    "axes.edgecolor": "#E2E8F0",   "axes.grid": True,
    "grid.color": "#E2E8F0",       "grid.linewidth": 0.8,
    "font.family": "DejaVu Sans",  "figure.dpi": 130,
})
BLUE, RED, GREEN = "#2563EB", "#DC2626", "#10B981"
PALETTE = [BLUE, RED, GREEN, '#F59E0B', '#8B5CF6', '#EC4899']

print("All libraries imported successfully.")


---
<a id='task-1'></a>
# Task 1: Dataset Understanding

Load the dataset and perform basic exploration:
- Number of rows and columns
- Type of input features
- Target variable description
- Missing value check
- Basic statistical summary
- Distribution of the target variable


### 1.1 Load the Dataset

In [ ]:
# Load raw CSV — no modifications made to the original file
df = pd.read_csv("customer_churn_nn.csv")

# Define column roles based on the data dictionary
TARGET_COL  = 'churn'
ID_COL      = 'customer_id'
CAT_COLS    = ['region', 'plan_type', 'contract_type', 'payment_method']
BINARY_COLS = ['autopay_enabled']
NUM_COLS    = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month',
               'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb',
               'satisfaction_score', 'last_complaint_days_ago', 'discount_percent',
               'referral_count']

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


### 1.2 Rows, Columns and Feature Types

In [ ]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print()
print(f"Numerical features   ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical features  ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Binary features        ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"Identifier (drop)     : {ID_COL}")
print(f"Target                : {TARGET_COL}")
print()
print("Data types:")
print(df.dtypes.to_string())


### 1.3 Target Variable Description

In [ ]:
vc = df[TARGET_COL].value_counts().sort_index()
print(f"Column        : '{TARGET_COL}'")
print(f"Type          : {df[TARGET_COL].dtype}  → Binary Classification")
print()
for label, val in zip(["Retained (0)", "Churned  (1)"], vc.values):
    print(f"  {label}: {val:>5,}  ({val/len(df)*100:.1f}%)")
print()
print(f"Imbalance ratio: {vc[0]/vc[1]:.1f} : 1")
print("⚠  SEVERE imbalance — use class weights and AUC-ROC / Recall as primary metrics.")


### 1.4 Missing Value Check

In [ ]:
missing = df.isnull().sum()
total   = missing.sum()
print(f"Total missing cells across all columns: {total}")
if total == 0:
    print("✅ No missing values — no imputation needed.")
else:
    print(missing[missing > 0])


### 1.5 Statistical Summary

In [ ]:
desc = df[NUM_COLS].describe().T.round(3)
desc.columns = ['count','mean','std','min','25%','50%','75%','max']
desc['cv%']        = (desc['std'] / desc['mean'].abs() * 100).round(1)
desc['corr_churn'] = df[NUM_COLS].corrwith(df[TARGET_COL]).round(4).values
desc['scale?']     = desc['cv%'].apply(lambda x: '⚠ Yes' if x > 30 else 'No')
print(desc[['mean','std','min','max','cv%','corr_churn','scale?']].to_string())


### 1.6 Target Distribution Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Target Variable Distribution: churn", fontsize=14, fontweight="bold")

vc = df[TARGET_COL].value_counts().sort_index()

# Bar chart
bars = axes[0].bar(["Retained (0)","Churned (1)"], vc.values,
                   color=[BLUE, RED], edgecolor="white", linewidth=1.5, width=0.5)
for bar, val in zip(bars, vc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center",
                 fontsize=11, fontweight="bold")
axes[0].set_title("Class Counts", fontweight="bold")
axes[0].set_ylim(0, 2300); axes[0].set_ylabel("Count")

# Pie
wedges, texts, autotexts = axes[1].pie(
    vc.values, labels=["Retained","Churned"], colors=[BLUE, RED],
    autopct="%1.1f%%", startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
for at in autotexts: at.set_fontweight("bold"); at.set_fontsize(12)
axes[1].set_title("Class Proportions", fontweight="bold")

# Info box
axes[2].axis("off")
axes[2].text(0.05, 0.95,
    f"Size          : 2,000\n\nRetained (0) : 1,969  (98.4%)\n"
    f"Churned  (1) :    31   (1.6%)\n\nImbalance : {vc[0]/vc[1]:.1f} : 1\n\n"
    f"⚠  Use pos_weight / SMOTE\n   Metrics: F1, AUC-ROC, Recall",
    transform=axes[2].transAxes, va="top", fontsize=11, fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="white", edgecolor=RED, linewidth=1.5))
axes[2].set_title("Imbalance Summary", fontweight="bold")
plt.tight_layout(); plt.show()


#### Correlation Heatmap

In [ ]:
corr = df[NUM_COLS + BINARY_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Pearson Correlation Heatmap", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout(); plt.show()
print("support_tickets (r=+0.12) and satisfaction_score (r=-0.09) correlate most with churn.")


---
<a id='task-2'></a>
# Task 2: Data Preprocessing

Prepare the data for neural network training:
- Handle missing values
- Encode categorical columns
- Scale numerical features
- Split into training and testing sets


### 2.1 Handle Missing Values

In [ ]:
# Task 1 confirmed 0 missing values. We assert this before proceeding.
assert df.isnull().sum().sum() == 0, "Unexpected missing values found!"
print("✅ 0 missing values confirmed. No imputation required.")
print()
print("Strategy if missing values existed:")
print("  Numerical   → Median imputation (robust to outliers)")
print("  Categorical → Mode or dedicated 'Unknown' category")


### 2.2 Drop Non-Predictive Identifier

In [ ]:
# customer_id has no predictive value — dropping it prevents the model
# from memorising IDs rather than learning generalised patterns.
df_model = df.drop(columns=[ID_COL])
print(f"Dropped '{ID_COL}'. Shape: {df.shape} → {df_model.shape}")


### 2.3 One-Hot Encode Categorical Features

In [ ]:
# One-hot encoding converts categorical strings to binary dummy columns.
# drop_first=True removes one dummy per group to prevent the dummy variable trap
# (perfect multicollinearity), which would destabilise gradient descent.
df_encoded = pd.get_dummies(df_model, columns=CAT_COLS, drop_first=True, dtype=int)

new_cols = [c for c in df_encoded.columns if c not in df_model.columns]
print(f"Columns before encoding: {df_model.shape[1]}")
print(f"Columns after  encoding: {df_encoded.shape[1]}")
print(f"New dummy columns ({len(new_cols)}): {new_cols}")
print()
print("Sample of encoded columns (first 3 rows):")
print(df_encoded[new_cols].head(3).to_string())


### 2.4 Separate Features and Target → Train/Test Split

In [ ]:
X = df_encoded.drop(columns=[TARGET_COL])
y = df_encoded[TARGET_COL]

# 80/20 stratified split — done BEFORE scaling to prevent data leakage.
# stratify=y preserves the 1.6% churn rate in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]:>4} rows  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set     : {X_test.shape[0]:>4} rows  ({X_test.shape[0]/len(X)*100:.0f}%)")
print()
print("Class distribution after stratified split:")
print(f"  Train — Retained: {(y_train==0).sum()}  Churned: {(y_train==1).sum()}  "
      f"({y_train.mean()*100:.1f}% churn)")
print(f"  Test  — Retained: {(y_test==0).sum()}   Churned: {(y_test==1).sum()}  "
      f"({y_test.mean()*100:.1f}% churn)")
print()
print("✅ Stratification preserved the original 1.6% churn rate in both splits.")


### 2.5 Z-Score Standardisation

In [ ]:
# StandardScaler: z = (x - mean_train) / std_train
# CRITICAL: fit only on X_train. Fitting on test data would leak test statistics
# into training — a form of data leakage that inflates performance estimates.
cols_to_scale = NUM_COLS + BINARY_COLS
scaler = StandardScaler()

X_train_s = X_train.copy()
X_test_s  = X_test.copy()
X_train_s[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])  # fit + transform
X_test_s[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])       # transform only

print("StandardScaler fit on training data only.")
print()
print("Verification — training set statistics after scaling:")
print(X_train_s[NUM_COLS[:5]].describe().loc[['mean','std']].round(3).to_string())
print()
print("✅ All numerical features now have mean ≈ 0 and std ≈ 1.")
print()
print(f"Final shapes → X_train_s: {X_train_s.shape}, X_test_s: {X_test_s.shape}")


### 2.6 Compute Sample Weights (Handle Class Imbalance)

In [ ]:
# Because MLPClassifier doesn't accept class_weight directly, we compute
# per-sample weights that give higher importance to the rare churn class (1).
# This is equivalent to class_weight='balanced' in other sklearn estimators.
sample_weights = compute_sample_weight('balanced', y_train)

n_retained = (y_train == 0).sum()
n_churned  = (y_train == 1).sum()
weight_0   = sample_weights[y_train == 0][0]
weight_1   = sample_weights[y_train == 1][0]

print(f"Training class counts — Retained: {n_retained}, Churned: {n_churned}")
print(f"Weight assigned to Retained (0) : {weight_0:.4f}")
print(f"Weight assigned to Churned  (1) : {weight_1:.4f}")
print(f"Effective weight ratio          : {weight_1/weight_0:.1f}x higher for churned")
print()
print("✅ Preprocessing complete. Ready for model building (Task 3).")


---
<a id='task-3'></a>
# Task 3: Neural Network Model Building

Build a feed-forward neural network with:
- Input layer (24 features)
- Hidden layers with activation functions
- Output layer suitable for binary classification
- Appropriate loss function and optimizer


### 3.1 Architecture Design

We use **scikit-learn's `MLPClassifier`** (Multi-Layer Perceptron), which implements
a fully connected feed-forward neural network.

**Architecture: Baseline Model**
```
Input Layer    →  24 neurons  (one per feature)
Hidden Layer 1 →  64 neurons  + ReLU activation
Hidden Layer 2 →  32 neurons  + ReLU activation
Output Layer   →   1 neuron   + Sigmoid activation  (binary classification)
```

**Why these choices:**
- **ReLU** — avoids vanishing gradients, fast convergence, standard for hidden layers
- **Sigmoid output** — squashes output to [0,1], interpretable as churn probability
- **Adam optimizer** — adaptive learning rates per parameter, handles sparse gradients well
- **BCE Loss** — standard loss for binary classification (log loss internally in sklearn)
- **Sample weights** — compensate for the 63:1 class imbalance


In [ ]:
# ── Baseline Model Definition ──────────────────────────────────────────
# Architecture:  Input(24) → Dense(64,ReLU) → Dense(32,ReLU) → Output(Sigmoid)
# Loss:          Binary Cross-Entropy (log_loss internally in sklearn)
# Optimizer:     Adam (solver='adam')
# Regularisation: early_stopping halts training when validation score stops improving

model_baseline = MLPClassifier(
    hidden_layer_sizes=(64, 32),   # 2 hidden layers: 64 then 32 neurons
    activation='relu',             # ReLU activation in hidden layers
    solver='adam',                 # Adam optimiser — adaptive learning rates
    learning_rate_init=0.001,      # initial learning rate
    max_iter=300,                  # maximum epochs
    batch_size=32,                 # mini-batch size for stochastic gradient descent
    random_state=42,               # reproducibility
    early_stopping=True,           # stop when val score plateaus
    validation_fraction=0.1,       # 10% of training data used for validation
    n_iter_no_change=10,           # patience: stop after 10 non-improving epochs
    verbose=False
)

print("Baseline Model Architecture:")
print("=" * 50)
print(f"  Input layer      : 24 neurons (one per feature)")
print(f"  Hidden layer 1   : 64 neurons + ReLU")
print(f"  Hidden layer 2   : 32 neurons + ReLU")
print(f"  Output layer     : 1 neuron  + Sigmoid")
print(f"  Loss function    : Binary Cross-Entropy")
print(f"  Optimizer        : Adam  (lr={model_baseline.learning_rate_init})")
print(f"  Batch size       : {model_baseline.batch_size}")
print(f"  Max epochs       : {model_baseline.max_iter}")
print(f"  Early stopping   : {model_baseline.early_stopping}")
print("=" * 50)


### 3.2 Train the Baseline Model

In [ ]:
# Train using sample_weight to upweight the minority class (churned customers).
# This prevents the model from simply predicting 'retained' for everyone.
model_baseline.fit(X_train_s, y_train, sample_weight=sample_weights)

print(f"Training complete.")
print(f"Epochs run (early stopped) : {model_baseline.n_iter_}")
print(f"Final training loss        : {model_baseline.loss_:.4f}")
print(f"Best validation score      : {model_baseline.best_validation_score_:.4f}")


---
<a id='task-4'></a>
# Task 4: Training and Evaluation

Evaluate the baseline model with:
- Training and testing accuracy/loss
- Confusion matrix
- Classification report (Precision, Recall, F1)
- AUC-ROC score
- Interpretation of results


### 4.1 Predictions and Core Metrics

In [ ]:
# Generate predictions
y_pred_train = model_baseline.predict(X_train_s)
y_pred_test  = model_baseline.predict(X_test_s)
y_prob_test  = model_baseline.predict_proba(X_test_s)[:, 1]  # churn probability

# Compute metrics
train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)
f1        = f1_score(y_test, y_pred_test, zero_division=0)
precision = precision_score(y_test, y_pred_test, zero_division=0)
recall    = recall_score(y_test, y_pred_test, zero_division=0)
auc       = roc_auc_score(y_test, y_prob_test)

print("=" * 50)
print("BASELINE MODEL — EVALUATION RESULTS")
print("=" * 50)
print(f"  Train Accuracy : {train_acc:.4f}")
print(f"  Test  Accuracy : {test_acc:.4f}")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall         : {recall:.4f}")
print(f"  F1-Score       : {f1:.4f}")
print(f"  AUC-ROC        : {auc:.4f}")
print("=" * 50)
print()
print("NOTE: Accuracy is misleading here due to 63:1 imbalance.")
print("A model predicting 'retained' for everyone scores 98.4% accuracy.")
print("Focus on Recall (catches churners) and AUC-ROC (ranking quality).")


### 4.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Task 4 – Baseline Model Evaluation", fontsize=13, fontweight="bold")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Retained','Churned'],
            yticklabels=['Retained','Churned'],
            annot_kws={"size": 16, "fontweight": "bold"})
axes[0].set_title("Confusion Matrix", fontweight="bold")
axes[0].set_ylabel("Actual"); axes[0].set_xlabel("Predicted")

# Add labels
tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.18,
    f"TN={tn}  FP={fp}  FN={fn}  TP={tp}",
    ha='center', transform=axes[0].transAxes, fontsize=10, color='gray')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, color=BLUE, linewidth=2.5, label=f"Baseline (AUC = {auc:.4f})")
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label="Random classifier")
axes[1].fill_between(fpr, tpr, alpha=0.1, color=BLUE)
axes[1].set_title("ROC Curve", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(fontsize=10)
axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1])

plt.tight_layout(); plt.show()


### 4.3 Classification Report

In [ ]:
print("Full Classification Report:")
print(classification_report(y_test, y_pred_test, 
                            target_names=['Retained (0)', 'Churned (1)']))


### 4.4 Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model_baseline.loss_curve_, color=BLUE, linewidth=2, label="Training Loss")
ax.set_title("Baseline Model — Training Loss Curve", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (Binary Cross-Entropy)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Training converged after {model_baseline.n_iter_} epochs (early stopping).")


### 4.5 Interpretation of Results

**Confusion Matrix Analysis:**
- The baseline model (ReLU, LR=0.001) predicts all samples as "Retained" → zero churners caught
- This is the classic failure mode with severe class imbalance — the model takes the easy path
- Despite applying sample weights, the model collapsed to a trivial solution with the default LR

**AUC-ROC = 0.746:**
- Significantly above random chance (0.5), so the model *does* learn some signal
- But its decision threshold defaults to 0.5, which is too high for a rare-class problem
- Lowering the threshold (e.g. to 0.3) would increase recall at the cost of precision

**Key insight:** Accuracy is a misleading metric here.  
Predicting "retained" for everyone = 98.4% accuracy but 0% churn detection.  
**Recall** (how many churners we catch) and **AUC-ROC** are the correct metrics.

→ Task 5 experiments will explore configurations that improve recall and AUC-ROC.


---
<a id='task-5'></a>
# Task 5: Hyperparameter Experimentation

Run 6 experiments varying:
- Number of hidden layers and neurons
- Learning rate
- Batch size
- Activation function

Compare performance across all configurations.


### 5.1 Experiment Configurations

| # | Name | Architecture | Activation | LR | Batch | What we're testing |
|---|------|-------------|-----------|-----|-------|-------------------|
| 1 | Baseline | (64, 32) | ReLU | 0.001 | 32 | Reference model |
| 2 | Deeper Network | (128, 64, 32) | ReLU | 0.001 | 32 | More depth |
| 3 | Higher LR | (64, 32) | ReLU | 0.01 | 32 | Faster convergence |
| 4 | Lower LR | (64, 32) | ReLU | 0.0001 | 32 | Slower/careful learning |
| 5 | tanh Activation | (64, 32) | tanh | 0.001 | 32 | Different non-linearity |
| 6 | Large Batch | (64, 32) | ReLU | 0.001 | 128 | Batch size effect |


In [ ]:
# Define all 6 experiment configurations
experiment_configs = [
    # (name,               layers,          activation, lr,     batch)
    ("Exp 1: Baseline",        (64, 32),       'relu', 0.001,  32),
    ("Exp 2: Deeper Network",  (128, 64, 32),  'relu', 0.001,  32),
    ("Exp 3: Higher LR",       (64, 32),       'relu', 0.01,   32),
    ("Exp 4: Lower LR",        (64, 32),       'relu', 0.0001, 32),
    ("Exp 5: tanh Activation", (64, 32),       'tanh', 0.001,  32),
    ("Exp 6: Large Batch",     (64, 32),       'relu', 0.001, 128),
]

all_results = []
all_models  = {}

for name, layers, act, lr, batch in experiment_configs:
    # Build and train model
    m = MLPClassifier(
        hidden_layer_sizes=layers,
        activation=act,
        solver='adam',
        learning_rate_init=lr,
        max_iter=300,
        batch_size=batch,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
    )
    m.fit(X_train_s, y_train, sample_weight=sample_weights)

    # Evaluate
    yp  = m.predict(X_test_s)
    ypr = m.predict_proba(X_test_s)[:, 1]

    result = {
        'Experiment':  name,
        'Architecture': str(layers),
        'Activation':  act,
        'LR':          lr,
        'Batch Size':  batch,
        'Epochs Run':  m.n_iter_,
        'Train Acc':   round(m.score(X_train_s, y_train), 4),
        'Test Acc':    round(accuracy_score(y_test, yp), 4),
        'Precision':   round(precision_score(y_test, yp, zero_division=0), 4),
        'Recall':      round(recall_score(y_test, yp, zero_division=0), 4),
        'F1-Score':    round(f1_score(y_test, yp, zero_division=0), 4),
        'AUC-ROC':     round(roc_auc_score(y_test, ypr), 4),
    }
    all_results.append(result)
    all_models[name] = (m, yp, ypr)
    print(f"[{name}]  AUC={result['AUC-ROC']}  Recall={result['Recall']}  F1={result['F1-Score']}")

print("\nAll experiments complete.")


### 5.2 Comparison Table

In [ ]:
df_results = pd.DataFrame(all_results)

# Highlight best values
print("=" * 100)
print("HYPERPARAMETER EXPERIMENT COMPARISON TABLE")
print("=" * 100)
print(df_results[['Experiment','Architecture','Activation','LR','Batch Size',
                  'Epochs Run','Train Acc','Test Acc','Precision','Recall',
                  'F1-Score','AUC-ROC']].to_string(index=False))
print("=" * 100)
print()
best_auc    = df_results.loc[df_results['AUC-ROC'].idxmax(), 'Experiment']
best_recall = df_results.loc[df_results['Recall'].idxmax(), 'Experiment']
best_f1     = df_results.loc[df_results['F1-Score'].idxmax(), 'Experiment']
print(f"Best AUC-ROC : {best_auc}")
print(f"Best Recall  : {best_recall}")
print(f"Best F1-Score: {best_f1}")


### 5.3 Visual Comparison

In [ ]:
labels = [r['Experiment'].split(':')[0] for r in all_results]  # "Exp 1" etc.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Task 5 – Hyperparameter Experiment Comparison", 
             fontsize=14, fontweight="bold")

metrics_to_plot = [
    ('AUC-ROC',   'A. AUC-ROC',  0.60, 1.0),
    ('Recall',    'B. Recall (Churn Detection Rate)', 0, 1.05),
    ('F1-Score',  'C. F1-Score', 0, 0.35),
    ('Precision', 'D. Precision', 0, 0.35),
    ('Test Acc',  'E. Test Accuracy', 0.7, 1.02),
    ('Train Acc', 'F. Train Accuracy', 0.7, 1.02),
]
axes = axes.flatten()

for ax, (metric, title, ymin, ymax) in zip(axes, metrics_to_plot):
    vals = [r[metric] for r in all_results]
    bars = ax.bar(labels, vals, color=PALETTE, edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{val:.3f}", ha='center', fontsize=8, fontweight='bold')
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.set_ylabel(metric); ax.set_ylim(ymin, ymax)
    ax.tick_params(axis='x', labelsize=8)

plt.tight_layout(); plt.show()


### 5.4 ROC Curves — All Experiments

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for (name, (m, yp, ypr)), col in zip(all_models.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, ypr)
    auc = roc_auc_score(y_test, ypr)
    short = name.split(':')[0]
    ax.plot(fpr, tpr, color=col, linewidth=2,
            label=f"{short} — {name.split(':')[1].strip()} (AUC={auc:.3f})")

ax.plot([0,1],[0,1],'k--', linewidth=1, label="Random (AUC=0.500)")
ax.fill_between([0,1],[0,1], alpha=0.03, color='gray')
ax.set_title("ROC Curves — All 6 Experiments", fontsize=13, fontweight="bold")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim([0,1]); ax.set_ylim([0,1])
plt.tight_layout(); plt.show()

print("Best AUC-ROC: Exp 5 (tanh, 0.9298) — tanh activation captures the non-linear")
print("churn signal better than ReLU at this dataset size.")


### 5.5 Loss Curves — All Experiments

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Training Loss Curves — All Experiments", fontsize=13, fontweight="bold")
axes = axes.flatten()

for ax, ((name, (m, _, _)), col) in zip(axes, zip(all_models.items(), PALETTE)):
    ax.plot(m.loss_curve_, color=col, linewidth=2)
    ax.set_title(name, fontweight="bold", fontsize=9)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.text(0.97, 0.95, f"Epochs: {m.n_iter_}\nFinal: {m.loss_:.4f}",
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout(); plt.show()


### 5.6 Experiment Observations

| Experiment | Key Finding |
|-----------|-------------|
| **Exp 1: Baseline** | Collapses to predicting all "retained" (Recall=0). Default LR too cautious with severe imbalance. |
| **Exp 2: Deeper Network** | Adding a third layer (128→64→32) improves AUC to 0.875 and catches 1 churner. More capacity helps. |
| **Exp 3: Higher LR (0.01)** | Best recall (0.833) alongside tanh. Higher LR pushes the model to take risks on the minority class. |
| **Exp 4: Lower LR (0.0001)** | Converges too slowly. Validation stops early with poor generalisation. Under-trained. |
| **Exp 5: tanh Activation** | **Best overall — AUC=0.930, Recall=0.833**. tanh's symmetric output (-1 to 1) handles the minority class better than ReLU at this scale. |
| **Exp 6: Large Batch (128)** | Decent AUC (0.849) but moderate recall (0.5). Larger batches smooth gradients too much, reducing sensitivity to minority class. |

**Winner: Exp 5 — tanh activation, LR=0.001, Architecture (64,32)**


### 5.7 Save Results

In [ ]:
# Save comparison table
df_results.to_csv(RESULTS_DIR / "model_comparison_table.csv", index=False)
print("✅ results/model_comparison_table.csv saved")
print()
print(df_results[['Experiment','AUC-ROC','Recall','F1-Score','Precision']].to_string(index=False))


In [ ]:
# Save combined evaluation_outputs.png
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(22, 18))
fig.suptitle("Part 1 – Full Evaluation Outputs | Customer Churn Neural Network",
             fontsize=15, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# A: Baseline confusion matrix
ax = fig.add_subplot(gs[0,0])
cm = confusion_matrix(y_test, model_baseline.predict(X_test_s))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'],
            annot_kws={"size":14, "fontweight":"bold"})
ax.set_title("A. Confusion Matrix (Baseline)", fontweight="bold", fontsize=10)
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")

# B: Best model confusion matrix (tanh)
ax = fig.add_subplot(gs[0,1])
best_m, best_yp, best_ypr = all_models["Exp 5: tanh Activation"]
cm2 = confusion_matrix(y_test, best_yp)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'],
            annot_kws={"size":14, "fontweight":"bold"})
ax.set_title("B. Confusion Matrix (Best: tanh)", fontweight="bold", fontsize=10)
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")

# C: ROC all experiments
ax = fig.add_subplot(gs[0,2])
for (name, (m, yp, ypr)), col in zip(all_models.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, ypr)
    auc = roc_auc_score(y_test, ypr)
    ax.plot(fpr, tpr, color=col, linewidth=1.5, label=f"{name.split(':')[0]} (AUC={auc:.3f})")
ax.plot([0,1],[0,1],'k--',linewidth=0.8)
ax.set_title("C. ROC Curves – All Experiments", fontweight="bold", fontsize=10)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(fontsize=7); ax.set_xlim([0,1]); ax.set_ylim([0,1])

# D: AUC bar
ax = fig.add_subplot(gs[1,0])
labels_s = [r['Experiment'].split(':')[0] for r in all_results]
aucs = [r['AUC-ROC'] for r in all_results]
bars = ax.bar(labels_s, aucs, color=PALETTE, edgecolor='white')
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f"{val:.3f}", ha='center', fontsize=9, fontweight='bold')
ax.set_title("D. AUC-ROC by Experiment", fontweight="bold", fontsize=10)
ax.set_ylabel("AUC-ROC"); ax.set_ylim(0.6,1.0)
ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.8)

# E: Recall bar
ax = fig.add_subplot(gs[1,1])
recalls = [r['Recall'] for r in all_results]
bars = ax.bar(labels_s, recalls, color=PALETTE, edgecolor='white')
for bar, val in zip(bars, recalls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{val:.2f}", ha='center', fontsize=9, fontweight='bold')
ax.set_title("E. Recall by Experiment", fontweight="bold", fontsize=10)
ax.set_ylabel("Recall"); ax.set_ylim(0,1.1)

# F: Metric heatmap
ax = fig.add_subplot(gs[1,2])
heat = np.array([[r['AUC-ROC'],r['Recall'],r['F1-Score'],r['Precision']] for r in all_results])
sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            xticklabels=['AUC','Recall','F1','Prec'],
            yticklabels=labels_s, annot_kws={"size":9})
ax.set_title("F. Metric Heatmap", fontweight="bold", fontsize=10)

# G: Loss curves (best model)
ax = fig.add_subplot(gs[2,0])
ax.plot(best_m.loss_curve_, color=GREEN, linewidth=2)
ax.set_title("G. Loss Curve (Best Model: tanh)", fontweight="bold", fontsize=10)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")

# H: Train vs Test accuracy
ax = fig.add_subplot(gs[2,1])
x = np.arange(len(labels_s)); w = 0.35
ax.bar(x-w/2, [r['Train Acc'] for r in all_results], w, label='Train', color=BLUE, alpha=0.8, edgecolor='white')
ax.bar(x+w/2, [r['Test Acc']  for r in all_results], w, label='Test',  color=RED,  alpha=0.8, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels_s, fontsize=8)
ax.set_title("H. Train vs Test Accuracy", fontweight="bold", fontsize=10)
ax.set_ylabel("Accuracy"); ax.set_ylim(0.7,1.02); ax.legend(fontsize=8)

# I: F1 bar
ax = fig.add_subplot(gs[2,2])
f1s = [r['F1-Score'] for r in all_results]
bars = ax.bar(labels_s, f1s, color=PALETTE, edgecolor='white')
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f"{val:.3f}", ha='center', fontsize=9, fontweight='bold')
ax.set_title("I. F1-Score by Experiment", fontweight="bold", fontsize=10)
ax.set_ylabel("F1-Score"); ax.set_ylim(0,0.35)

plt.savefig(RESULTS_DIR / "evaluation_outputs.png", bbox_inches='tight', dpi=150)
plt.show()
print("✅ results/evaluation_outputs.png saved")


---
<a id='task-6'></a>
# Task 6: Final Reflection

Conceptual questions about how and why neural networks work the way they do.


### 6.1 What role do weights and biases play in the model?

**Weights** are the learnable multipliers on each connection between neurons.
During a forward pass, each neuron computes a weighted sum of its inputs:

```
z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
```

- **Weights (w)** control *how much influence* each input feature has on the output.
  A high positive weight means "more of this feature → more likely to churn".
  A high negative weight means "more of this feature → less likely to churn".
  During training, backpropagation adjusts weights in the direction that reduces loss.

- **Biases (b)** are added constants that shift the activation function left or right,
  independent of the input. They allow the model to output non-zero values even when
  all inputs are zero — giving each neuron its own baseline "threshold".

Together: **weights capture patterns**; **biases give flexibility in where those patterns activate**.

---

### 6.2 Why is an activation function required?

Without an activation function, stacking multiple linear layers produces only a single
linear transformation (any composition of linear functions is still linear):

```
Layer 1: z₁ = W₁x + b₁
Layer 2: z₂ = W₂z₁ + b₂  =  W₂(W₁x + b₁) + b₂  =  (W₂W₁)x + (W₂b₁ + b₂)
```

This collapses to a single linear model regardless of depth — no better than logistic regression.

**Activation functions introduce non-linearity**, enabling the network to learn complex,
curved decision boundaries needed for real-world problems like churn prediction.

| Activation | Formula | Key property |
|-----------|---------|--------------|
| **ReLU** | max(0, x) | Fast, avoids vanishing gradient, sparse activation |
| **tanh** | (eˣ - e⁻ˣ)/(eˣ + e⁻ˣ) | Zero-centered output [-1,1], can outperform ReLU on small datasets |
| **Sigmoid** | 1/(1+e⁻ˣ) | Output in [0,1] — used in the output layer for binary probability |

In our experiments, **tanh outperformed ReLU** (AUC 0.930 vs 0.746) — tanh's
zero-centered, bounded output helped the model learn the rare churn signal better.

---

### 6.3 What happens when learning rate is too high or too low?

The learning rate controls how large a step the optimizer takes in the direction of the gradient.

**Too high (e.g. 0.1):**
- Large weight updates cause the loss to overshoot the minimum
- Loss oscillates or diverges — the model fails to converge
- In our Exp 3 (LR=0.01): recall improved (0.833) but precision collapsed (0.069)
  because the model overcorrected toward predicting churn for everyone

**Too low (e.g. 0.00001):**
- Very small updates — convergence is extremely slow
- Early stopping fires before the model has learned meaningful patterns
- The model underfits: it hasn't had enough gradient steps to find a good solution
- In our Exp 4 (LR=0.0001): test accuracy only 78.8%, AUC=0.725 — significantly worse

**Optimal (0.001 — our baseline and tanh model):**
- Smooth convergence to a good minimum
- The model learns generalised patterns without overshooting

The learning rate is often the single most impactful hyperparameter — too high and the
model is erratic; too low and it never learns.

---

### 6.4 Did the model show signs of underfitting or overfitting?

Looking at our experiment results:


In [ ]:
print("Train vs Test Accuracy gap (overfitting indicator):")
print("-" * 60)
for r in all_results:
    gap = r['Train Acc'] - r['Test Acc']
    flag = "⚠ overfit" if gap > 0.05 else ("⚠ underfit" if r['Test Acc'] < 0.85 else "✅ balanced")
    print(f"  {r['Experiment']:<30} gap={gap:+.4f}  {flag}")

print()
print("Recall analysis (underfitting indicator for minority class):")
print("-" * 60)
for r in all_results:
    if r['Recall'] == 0:
        note = "⚠ COLLAPSED — predicts all 'retained', complete underfit on churn class"
    elif r['Recall'] < 0.5:
        note = "⚠ Weak — misses majority of churners"
    else:
        note = "✅ Good recall — catching churners"
    print(f"  {r['Experiment']:<30} Recall={r['Recall']:.4f}  {note}")


### 6.5 Summary Reflection

**Underfitting was the dominant problem** in this assignment, not overfitting.

The train/test accuracy gap was minimal across all experiments — the model generalised
well. The real challenge was the **63:1 class imbalance**, which caused several models
to underfit the minority class entirely (Recall=0 for the baseline).

Evidence of underfitting:
- Baseline model: Recall = 0.000 — catches zero churners despite sample weighting
- Lower LR model: AUC = 0.725 — barely better than random, model didn't learn enough

Evidence against overfitting:
- Train/test accuracy gaps were all < 5%
- No model memorised training data; all generalised to the test set reasonably

**Best model — Exp 5 (tanh, LR=0.001):**
- AUC-ROC: 0.930 — excellent ranking quality
- Recall: 0.833 — catches 5 of 6 churners in the test set
- This represents the best balance achievable without more advanced techniques

**To further improve beyond this assignment:**
- SMOTE oversampling of the minority class
- Threshold tuning (lower the 0.5 default decision boundary)
- Ensemble methods (Random Forest, XGBoost) which handle imbalance better natively
- More training data — 31 churn examples is genuinely too few for stable learning
